# Questão 1: Pré-Treinamento Continuado de LLM
## Dataset: DOMPI-2025 | Modelo: Qwen2.5-0.5B

**Disciplina:** Tópicos em Inteligência Artificial  
**Aluno:** Allycia, Heverton, Izaías, Oscar

---

Este notebook realiza o **pré-treinamento continuado** do modelo `Qwen/Qwen2.5-0.5B` usando o corpus DOM-PI 2025 (Diário Oficial dos Municípios do Piauí). O experimento avalia se o modelo melhora sua capacidade de modelar textos do domínio jurídico/administrativo piauiense após o treinamento.

**Métricas avaliadas:** Perplexidade, Entropia Cruzada, Acurácia de Previsão de Tokens

## 1. Instalação das Dependências

In [ ]:
# Ambiente Kaggle: usamos a stack pre-instalada (torch, torchvision, transformers,
# datasets, accelerate) exatamente como vem na imagem, que ja e consistente entre si.
# NAO reinstale nem remova torch/torchvision: foi isso que quebrava o import do
# transformers nas tentativas anteriores.
# IMPORTANTE: se der erro de import, faca Run > Factory reset (nao basta clicar Run
# de novo: o kernel guarda o estado quebrado das execucoes anteriores).
print("Usando a stack pre-instalada do Kaggle (sem reinstalar torch/transformers).")

In [ ]:
# No Kaggle nao e preciso instalar nada: transformers, datasets, accelerate,
# tokenizers, safetensors, numpy e regex ja vem na imagem, em versoes consistentes
# com o torch/torchvision. Mexer nesses pacotes foi o que causou os erros de import.
# A instalacao fica desligada de proposito.
print("Dependencias ja presentes no Kaggle; nada a instalar.")

In [ ]:
'''
# Com suporte a GPU (CUDA 12.1):

import subprocess, sys


# Limpa tudo relacionado a transformers do ambiente
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "transformers", "tokenizers", "accelerate", "huggingface_hub"],
               capture_output=True)

# Reinstala tudo junto numa versão consistente
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.47.1",
                "tokenizers==0.21.0",
                "accelerate==1.2.1",
                "huggingface_hub==0.27.0",
                "--no-deps"], capture_output=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "safetensors", "datasets", "numpy"], capture_output=True)

print("Instalação concluída.")
'''

'\n# Com suporte a GPU (CUDA 12.1):\n\nimport subprocess, sys\n\n\n# Limpa tudo relacionado a transformers do ambiente\nsubprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",\n                "transformers", "tokenizers", "accelerate", "huggingface_hub"],\n               capture_output=True)\n\n# Reinstala tudo junto numa versão consistente\nsubprocess.run([sys.executable, "-m", "pip", "install", "-q",\n                "transformers==4.47.1",\n                "tokenizers==0.21.0",\n                "accelerate==1.2.1",\n                "huggingface_hub==0.27.0",\n                "--no-deps"], capture_output=True)\n\nsubprocess.run([sys.executable, "-m", "pip", "install", "-q",\n                "safetensors", "datasets", "numpy"], capture_output=True)\n\nprint("Instalação concluída.")\n'

In [ ]:
#import transformers
#print(transformers.__version__)

## 2. Imports e Configuração do Ambiente

In [ ]:
import os
# Usa apenas 1 GPU. Com T4 x2 o Trainer ativa DataParallel, que junta os logits
# (vocab 151k) na GPU 0 e estoura a memoria (OutOfMemoryError no gather). O 0.5B
# cabe folgado numa unica T4. Precisa vir ANTES do "import torch".
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
import math
import re
import random
import json
import copy
import numpy as np
from collections import Counter
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from torch.utils.data import Dataset

# Reproducibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memória disponível: {mem_gb:.1f} GB')
    print(f'GPUs visíveis: {torch.cuda.device_count()}')

## 3. Carregamento do Dataset DOMPI-2025

O dataset foi publicado no HuggingFace pelo Grupo 01 (Gutemberg) e contém ~77 mil documentos extraídos via OCR do Diário Oficial dos Municípios do Piauí.

In [ ]:
TERRITORIOS = [
    'carnaubais',
    'tabuleiros_alto_parnaiba',
    'planice_litoran',
    'entre_rios',
    'vale_do_sambito',
    'vale_dos_rios_piaui_e_itaueiras',
    'cocais',
    'mangabeiras',
    'serra_da_capivara',
    'vale_do_rio_guaribas',
    'chapada_vale_do_rio_itaim',
    'vale_do_caninde'
]

print('Carregando DOMPI-2025 do HuggingFace...')
splits = []
for t in TERRITORIOS:
    print(f'  -> {t}')
    ds = load_dataset('gutoportelaa/DOMPI-2025', name='raw', split=t)
    splits.append(ds)

dataset = concatenate_datasets(splits)
print(f'\nTotal de documentos: {len(dataset):,}')
print(f'Colunas: {dataset.column_names}')

Carregando DOMPI-2025 do HuggingFace...
  -> carnaubais
  -> tabuleiros_alto_parnaiba
  -> planice_litoran
  -> entre_rios
  -> vale_do_sambito
  -> vale_dos_rios_piaui_e_itaueiras
  -> cocais
  -> mangabeiras
  -> serra_da_capivara
  -> vale_do_rio_guaribas
  -> chapada_vale_do_rio_itaim
  -> vale_do_caninde

Total de documentos: 77,337
Colunas: ['id_publicacao', 'territorio', 'municipio', 'tipo_ato', 'data_publicacao', 'ano', 'extrator', 'texto', 'n_chars', 'paginas', 'extraido_em', 'extracao_segundos']


In [ ]:
# Exploração rápida do dataset
print('=== Amostra de documento ===')
doc = dataset[0]
for campo in ['municipio', 'territorio', 'tipo_ato', 'data_publicacao', 'extrator', 'n_chars']:
    print(f'  {campo}: {doc[campo]}')
print(f'  texto (trecho): {doc["texto"][:200]}...')

print('\n=== Tipos de ato mais frequentes ===')
for tipo, n in Counter(dataset['tipo_ato']).most_common(8):
    print(f'  {tipo}: {n:,}')

=== Amostra de documento ===
  municipio: A  Favor  Ou  Contra  Determinado
  territorio: carnaubais
  tipo_ato: Decreto
  data_publicacao: 2025
  extrator: paddle-cuda
  n_chars: 25776
  texto (trecho): 245
Ano Xxill - Teresina (Pl) - Segunda-Feira, 01 de Setembro de 2025 - Edicao VCccxcV
STADO DO Piauí
Piauí
Av.Sao
NPJ: 01.878.514/0001-07
CNPJ:01.878.514/0001-07
E-mail: camaradejuazeiro@hotmail.com
...

=== Tipos de ato mais frequentes ===
  Portaria: 19,514
  LRF: 13,090
  Decreto: 10,394
  Licitação: 9,961
  Edital: 6,971
  Lei: 6,734
  Contrato: 4,460
  Resolução: 1,223


## 4. Pré-processamento

O dataset está na versão `raw` (extração bruta por OCR). Aplicamos limpeza básica para reduzir ruídos antes de tokenizar.

In [ ]:
def limpar_texto(t):
    if not t:
        return ''
    t = re.sub(r'\n{3,}', '\n\n', t)          # colapsa quebras de linha excessivas
    t = re.sub(r' {2,}', ' ', t)               # remove espaços duplicados
    t = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', t)  # chars de controle
    return t.strip()

MIN_CHARS = 200
MAX_CHARS = 50_000

textos_brutos = dataset['texto']
textos = [
    limpar_texto(t)
    for t in textos_brutos
    if t and MIN_CHARS <= len(t) <= MAX_CHARS
]

random.shuffle(textos)

# Usa todo o corpus filtrado no treino, segurando N_AVALIACAO docs para avaliacao
N_AVALIACAO = 1000

textos_eval   = textos[:N_AVALIACAO]
textos_treino = textos[N_AVALIACAO:]

print(f'Antes da filtragem : {len(textos_brutos):,}')
print(f'Após filtragem     : {len(textos):,}')
print(f'Treino             : {len(textos_treino):,}')
print(f'Avaliação          : {len(textos_eval):,}')

In [ ]:
# Recupera os metadados dos textos que entraram no treino
# Cruza os textos limpos com o dataset original, iterando por colunas (rapido)

textos_treino_set = set(textos_treino)

territorios_treino = []
municipios_treino = []

for txt, terr, muni in zip(dataset['texto'], dataset['territorio'], dataset['municipio']):
    if not txt:
        continue
    if limpar_texto(txt) in textos_treino_set:
        territorios_treino.append(terr)
        municipios_treino.append(muni)

print(f"=== Distribuição dos {len(territorios_treino)} textos de treino ===")
print("\nPor território:")
for t, n in Counter(territorios_treino).most_common():
    pct = n / len(territorios_treino) * 100
    print(f"  {t:<40} {n:>5} ({pct:.1f}%)")

print(f"\nPor município (top 10):")
for m, n in Counter(municipios_treino).most_common(10):
    print(f"  {m:<40} {n:>5}")

print(f"\nTotal territórios representados: {len(set(territorios_treino))}")
print(f"Total municípios representados : {len(set(municipios_treino))}")

## 5. Carregamento do Modelo Base

Usamos o `Qwen/Qwen2.5-0.5B` (~494M parâmetros), um modelo multilíngue que já cobre português. Partir de um modelo já competente em português torna o pré-treino continuado mais eficiente do que partir de um modelo só em inglês.

In [ ]:
#MODEL_NAME = 'pierreguillou/gpt2-small-portuguese'
#MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
#MODEL_NAME = 'llama3.2:1b'
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'

MAX_LEN    =  256  #512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # garante um token de padding para o collator

# Carrega em fp32: o treino usa fp16=True (mixed precision com GradScaler), que exige
# pesos mestres em fp32. Sem isso, o transformers novo carrega o Qwen em bfloat16 e o
# GradScaler do fp16 quebra (NotImplementedError ... not implemented for 'BFloat16').
# A T4 nao tem bf16 em hardware, entao fp32 + fp16 autocast e o caminho certo.
model_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model_base.to(device)

n_params = sum(p.numel() for p in model_base.parameters())
print(f'Modelo carregado: {MODEL_NAME}')
print(f'Parâmetros: {n_params/1e6:.1f}M')
print(f'Vocabulário: {tokenizer.vocab_size:,} tokens')

## 6. Função de Avaliação (Métricas)

As três métricas são calculadas sobre o mesmo conjunto de avaliação, antes e depois do treinamento, para comparação direta.

- **Entropia Cruzada**: quão longe o modelo erra ao prever o próximo token. Menor = melhor.
- **Perplexidade** = e^(entropia cruzada). Intuitivamente: "quantas opções o modelo considera a cada token".
- **Acurácia de Tokens**: % de tokens em que o modelo acerta exatamente qual é o próximo.

In [ ]:
def calcular_metricas(modelo, textos, batch_size=8):
    modelo.eval()
    total_loss   = 0.0
    total_tokens = 0
    total_certos = 0
    total_shift  = 0

    for i in range(0, len(textos), batch_size):
        batch = textos[i : i + batch_size]

        enc = tokenizer(
            batch,
            return_tensors='pt',
            max_length=MAX_LEN,
            truncation=True,
            padding=True
        )
        input_ids      = enc['input_ids'].to(device)
        attention_mask = enc['attention_mask'].to(device)

        # Tokens de padding recebem label -100 (ignorados no loss)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        with torch.no_grad():
            out = modelo(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        n_tok = attention_mask.sum().item()
        total_loss   += out.loss.item() * n_tok
        total_tokens += n_tok

        # Acurácia: pred[t] vs token[t+1]
        preds      = out.logits.argmax(dim=-1)           # (B, L)
        mask_shift = attention_mask[:, 1:].bool()         # ignora padding
        certos     = (preds[:, :-1] == input_ids[:, 1:])[mask_shift].sum().item()
        total_certos += certos
        total_shift  += mask_shift.sum().item()

    ec  = total_loss / total_tokens
    ppl = math.exp(ec)
    acc = total_certos / total_shift

    return {'entropia_cruzada': round(ec, 4),
            'perplexidade':     round(ppl, 2),
            'acuracia_tokens':  round(acc * 100, 2)}

In [ ]:
benchmark = [
    # Entidades e Identificadores
    {'id':1,'cat':'Entidades','pergunta':'Qual é o CNPJ da Prefeitura Municipal de Aroazes?','resposta':'06.554.984/0001-39'},
    {'id':2,'cat':'Entidades','pergunta':'Qual é o CEP da Prefeitura Municipal de Aroazes?','resposta':'64.310-000'},
    {'id':3,'cat':'Entidades','pergunta':'Qual é o endereço da Prefeitura Municipal de Aroazes?','resposta':'Av. 27 de Fevereiro, 691, Centro'},
    {'id':4,'cat':'Entidades','pergunta':'Qual é o CNPJ da Câmara Municipal de Juazeiro do Piauí?','resposta':'01.878.514/0001-07'},
    {'id':5,'cat':'Entidades','pergunta':'Qual é o CNPJ da Prefeitura Municipal de Assunção do Piauí?','resposta':'01.612.561/0001-04'},
    # Atos Administrativos
    {'id':6,'cat':'Atos Adm.','pergunta':'Quais são os principais tipos de atos publicados no DOM-PI?','resposta':'Decreto, Portaria, Licitação, Edital, Lei, Ata, LRF'},
    {'id':7,'cat':'Atos Adm.','pergunta':'O que é um Relatório de Gestão Fiscal (RGF) e qual lei o regulamenta?','resposta':'Documento obrigatório de situação financeira regulamentado pela LRF (art. 55 da LC 101/2000)'},
    {'id':8,'cat':'Atos Adm.','pergunta':'O que é uma Dispensa Eletrônica no contexto de licitações?','resposta':'Contratação direta sem licitação realizada por meio eletrônico para aquisições de menor valor'},
    {'id':9,'cat':'Atos Adm.','pergunta':'O que é um Termo de Rescisão/Distrato de Contrato Administrativo?','resposta':'Documento que formaliza o encerramento antecipado e amigável de um contrato administrativo'},
    {'id':10,'cat':'Atos Adm.','pergunta':'O que é uma Ata de Registro de Preços?','resposta':'Documento que formaliza preços registrados após licitação, permitindo compras futuras sem nova licitação'},
    # Informações Territoriais
    {'id':11,'cat':'Territorial','pergunta':'Quantos territórios compõem o dataset DOMPI-2025?','resposta':'12 territórios'},
    {'id':12,'cat':'Territorial','pergunta':'Qual território possui o maior número de documentos no DOMPI-2025?','resposta':'Cocais, com 12.188 documentos'},
    {'id':13,'cat':'Territorial','pergunta':'Em qual cidade são publicados os Diários Oficiais dos Municípios do Piauí?','resposta':'Teresina (PI)'},
    {'id':14,'cat':'Territorial','pergunta':'Qual é o CEP da Prefeitura Municipal de Anísio de Abreu?','resposta':'64.780-000'},
    {'id':15,'cat':'Territorial','pergunta':'Qual é o e-mail da Prefeitura Municipal de Anísio de Abreu?','resposta':'pmanisiodeabreupi@gmail.com'},
    # Processos Administrativos
    {'id':16,'cat':'Processos','pergunta':'O que é o Pregão Eletrônico no contexto das licitações municipais?','resposta':'Modalidade licitatória realizada eletronicamente em que fornecedores competem por lances'},
    {'id':17,'cat':'Processos','pergunta':'O que é o Demonstrativo da Dívida Consolidada no RGF?','resposta':'RGF Anexo 2: apresenta o total das obrigações financeiras do município conforme a LRF'},
    {'id':18,'cat':'Processos','pergunta':'Quais motivos constituem causa para extinção de contrato administrativo?','resposta':'Descumprimento de cláusulas contratuais, irregularidade fiscal, entre outros (Art. 137 da Lei 14.133/2021)'},
    {'id':19,'cat':'Processos','pergunta':'O que é a Comissão Permanente de Licitação?','resposta':'Órgão colegiado da prefeitura responsável por conduzir e fiscalizar processos licitatórios'},
    {'id':20,'cat':'Processos','pergunta':'O que é uma Errata em publicação de diário oficial?','resposta':'Publicação que corrige erro material contido em publicação anterior'},
    # Legislação
    {'id':21,'cat':'Legislação','pergunta':'Qual lei federal regulamenta as licitações e contratos municipais atualmente?','resposta':'Lei Federal nº 14.133/2021 (Nova Lei de Licitações), que substituiu a Lei nº 8.666/1993'},
    {'id':22,'cat':'Legislação','pergunta':'O que significa a sigla LRF?','resposta':'Lei de Responsabilidade Fiscal — Lei Complementar nº 101/2000'},
    {'id':23,'cat':'Legislação','pergunta':'O que é o Orçamento Fiscal e da Seguridade Social na LOA?','resposta':'Os dois orçamentos da LOA: fiscal (governo geral) e seguridade social (saúde, previdência, assistência)'},
    # Sobre o Dataset
    {'id':24,'cat':'Dataset','pergunta':'Quais ferramentas foram usadas para extrair texto dos PDFs no DOMPI-2025?','resposta':'PaddleOCR-GPU (portarias/decretos), Docling (tabelas/relatórios) e PyMuPDF (texto nativo)'},
    {'id':25,'cat':'Dataset','pergunta':'Qual é o volume total de documentos e o período coberto pelo DOMPI-2025?','resposta':'77.337 documentos cobrindo publicações de 2025 do DOM-PI'},
]

#import json # Certifique-se de que isso está no topo do seu script/notebook

def rodar_benchmark(model, benchmark_list, tokenizer, device, max_new_tokens=100):
    model.eval()
    resultados = []
    for item in benchmark_list:
        inputs = tokenizer(item['pergunta'], return_tensors='pt').to(device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False
            )
        resposta_gerada = tokenizer.decode(
            output[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()

        resultados.append({
            'id': item['id'],
            'cat': item['cat'],
            'pergunta': item['pergunta'],
            'resposta_esperada': item['resposta'],
            'resposta_gerada': resposta_gerada
        })
    return resultados

print("Rodando benchmark no modelo BASE (antes do treinamento)...")
benchmark_antes = rodar_benchmark(model_base, benchmark, tokenizer, device)

print(f"\n{'ID':<5} {'Cat':<12} {'Esperada':<40} {'Gerada'}")
print('-' * 100)
for r in benchmark_antes:
    print(f"{r['id']:<5} {r['cat']:<12} {r['resposta_esperada']:<40} {r['resposta_gerada'][:60]}")

# ---------------------------------------------------------
# CÓDIGO ADICIONADO PARA SALVAR EM ARQUIVO JSON
# ---------------------------------------------------------
nome_arquivo = 'resultados_benchmark_base.json'

with open(nome_arquivo, 'w', encoding='utf-8') as f:
    json.dump(benchmark_antes, f, ensure_ascii=False, indent=4)

print(f"\nResultados do modelo base salvos no arquivo: {nome_arquivo}")

## 7. Avaliação ANTES do Pré-Treinamento (Baseline)

In [ ]:
print('Calculando métricas do modelo base...')
metricas_antes = calcular_metricas(model_base, textos_eval)

print('\n=== ANTES do Pré-Treinamento ===')
print(f"  Entropia Cruzada : {metricas_antes['entropia_cruzada']}")
print(f"  Perplexidade     : {metricas_antes['perplexidade']}")
print(f"  Acurácia Tokens  : {metricas_antes['acuracia_tokens']}%")

Calculando métricas do modelo base...

=== ANTES do Pré-Treinamento ===
  Entropia Cruzada : 3.0646
  Perplexidade     : 21.43
  Acurácia Tokens  : 42.4%


## 8. Preparação do Dataset para Treinamento

In [ ]:
class DiarioDataset(Dataset):
    def __init__(self, textos):
        self.enc = tokenizer(
            textos,
            max_length=MAX_LEN,
            truncation=True
        )

    def __len__(self):
        return len(self.enc['input_ids'])

    def __getitem__(self, idx):
        return {'input_ids': self.enc['input_ids'][idx]}

print('Tokenizando dados de treino...')
train_ds = DiarioDataset(textos_treino)
print('Tokenizando dados de avaliação (Trainer)...')
eval_ds  = DiarioDataset(textos_eval)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(f'Treino: {len(train_ds):,} | Avaliação: {len(eval_ds):,}')

## 9. Pré-Treinamento Continuado

O modelo base é copiado antes do treino para garantir que `model_base` permaneça intacto para comparação. O treinamento acontece sobre `model_ft` (fine-tuned).

In [ ]:
# Copia o modelo base para não sobrescrever os pesos originais
model_ft = copy.deepcopy(model_base)
model_ft.to(device)
model_ft.config.use_cache = False

OUTPUT_DIR = 'dompi_qwen25'
os.makedirs(OUTPUT_DIR, exist_ok=True)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=2,
    per_device_train_batch_size=4, #
    per_device_eval_batch_size=4, #
    gradient_accumulation_steps=8,      # batch efetivo = 16
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},

    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',

    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    fp16=(device == 'cuda'),
    dataloader_num_workers=2,

    logging_steps=50,
    report_to='none'
)

trainer = Trainer(
    model=model_ft,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
)

print('Iniciando pré-treinamento continuado...')
print(f'  Épocas: {args.num_train_epochs}')
print(f'  Batch efetivo: {args.per_device_train_batch_size * args.gradient_accumulation_steps}')
print(f'  Learning rate: {args.learning_rate}\n')

resultado = trainer.train()

print('\nTreinamento concluído!')
print(f"  Loss final de treino : {resultado.training_loss:.4f}")
print(f"  Tempo total          : {resultado.metrics['train_runtime']:.0f}s")

## 10. Avaliação APÓS o Pré-Treinamento

Carregamos explicitamente o **melhor checkpoint** salvo pelo Trainer (selecionado pela menor `eval_loss`), garantindo que avaliamos o melhor estado do modelo treinado, e não o estado do último step.

In [ ]:
# Identifica o melhor checkpoint
checkpoints = sorted(
    [d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-')],
    key=lambda x: int(x.split('-')[-1])
)
print(f'Checkpoints encontrados: {checkpoints}')

melhor_ckpt = trainer.state.best_model_checkpoint
if melhor_ckpt is None:
    melhor_ckpt = os.path.join(OUTPUT_DIR, checkpoints[-1])
print(f'Melhor checkpoint: {melhor_ckpt}')

# Carrega o melhor checkpoint em fp32 (mesmo dtype do baseline) para a comparacao de metricas ficar consistente
model_avaliacao = AutoModelForCausalLM.from_pretrained(melhor_ckpt, torch_dtype=torch.float32)
model_avaliacao.to(device)

print('\nCalculando métricas do modelo treinado...')
metricas_depois = calcular_metricas(model_avaliacao, textos_eval)

print('\n=== DEPOIS do Pré-Treinamento ===')
print(f"  Entropia Cruzada : {metricas_depois['entropia_cruzada']}")
print(f"  Perplexidade     : {metricas_depois['perplexidade']}")
print(f"  Acurácia Tokens  : {metricas_depois['acuracia_tokens']}%")

## 11. Comparação dos Resultados

In [ ]:
print('=' * 62)
print('     COMPARAÇÃO: ANTES vs. DEPOIS DO PRÉ-TREINAMENTO')
print('=' * 62)
print(f"{'Métrica':<28} {'Antes':>10} {'Depois':>10} {'Δ':>10}")
print('-' * 62)

d_ec  = metricas_depois['entropia_cruzada'] - metricas_antes['entropia_cruzada']
d_ppl = metricas_depois['perplexidade']     - metricas_antes['perplexidade']
d_acc = metricas_depois['acuracia_tokens']  - metricas_antes['acuracia_tokens']

print(f"{'Entropia Cruzada':<28} {metricas_antes['entropia_cruzada']:>10.4f} {metricas_depois['entropia_cruzada']:>10.4f} {d_ec:>+10.4f}")
print(f"{'Perplexidade (PPL)':<28} {metricas_antes['perplexidade']:>10.2f} {metricas_depois['perplexidade']:>10.2f} {d_ppl:>+10.2f}")
print(f"{'Acurácia de Tokens (%)':<28} {metricas_antes['acuracia_tokens']:>10.2f} {metricas_depois['acuracia_tokens']:>10.2f} {d_acc:>+10.2f}")
print('=' * 62)

var_ppl = (d_ppl / metricas_antes['perplexidade']) * 100
print(f'\nVariação relativa da Perplexidade: {var_ppl:+.1f}%')
if d_ppl < 0:
    print('✓ Perplexidade REDUZIU — modelo melhorou no domínio.')
else:
    print('✗ Perplexidade aumentou — verifique os checkpoints.')

     COMPARAÇÃO: ANTES vs. DEPOIS DO PRÉ-TREINAMENTO
Métrica                           Antes     Depois          Δ
--------------------------------------------------------------
Entropia Cruzada                 3.9262     1.8038    -2.1224
Perplexidade (PPL)                50.71       6.07     -44.64
Acurácia de Tokens (%)            35.81      64.08     +28.27

Variação relativa da Perplexidade: -88.0%
✓ Perplexidade REDUZIU — modelo melhorou no domínio.


## 12. Análise Qualitativa — Geração de Texto

Comparamos o texto gerado pelo modelo base e pelo modelo treinado para os mesmos prompts, evidenciando a adaptação ao domínio.

In [ ]:
def gerar(modelo, prompt, max_new=80):
    modelo.eval()
    inp = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = modelo.generate(
            **inp,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

prompts = [
    'O Prefeito Municipal, no uso de suas atribuições legais,',
    'PORTARIA Nº 001/2025. Nomeia servidor para o cargo de',
    'EXTRATO DE CONTRATO. Contratante: Prefeitura Municipal de'
]

for p in prompts:
    print(f'PROMPT : {p}')
    print(f'BASE   : {gerar(model_base, p)}')
    print(f'TREINO : {gerar(model_avaliacao, p)}')
    print('-' * 70)

PROMPT : O Prefeito Municipal, no uso de suas atribuições legais,
BASE   : O Prefeito Municipal, no uso de suas atribuições legais, e a Lei nº 9.790/96 que dispõe sobre os bens da União.

O município foi criado pela Lei Estadual nº 10.586, de 28 de junho de 1986.

Os símbolos oficiais do município são o brasão, a bandeira e o hino municipal.

O município é sede de uma unidade federativa brasileira, com capital em Brasília, Distrito Federal, além
TREINO : O Prefeito Municipal, no uso de suas atribuições legais, resolve:
Art. 1 - Nomear a Sra. FRANCISCO DE SOUSA SOARES, CPF sob o n. 032.568.373-87, para exercer o cargo de COORDENADOR,
inscrita no CPF sob o n. 036.844.273-51, lotada na Secretaria Municipal de
----------------------------------------------------------------------
PROMPT : PORTARIA Nº 001/2025. Nomeia servidor para o cargo de
BASE   : PORTARIA Nº 001/2025. Nomeia servidor para o cargo de subdiretor da Escola Técnica Federal do Rio de Janeiro (ETFRJ).

O Laboratório de Quími

## 13. Benchmark — 25 Pares Pergunta / Resposta de Referência

In [ ]:
print(f'Benchmark com {len(benchmark)} pares pergunta-resposta.\n')
print(f'{"#":<4} {"Categoria":<14} {"Pergunta":<55} {"Resposta"}')
print('-' * 130)
for q in benchmark:
    p = q['pergunta'][:53] + '..' if len(q['pergunta']) > 55 else q['pergunta']
    r = q['resposta'][:45] + '..' if len(q['resposta']) > 47 else q['resposta']
    print(f"{q['id']:<4} {q['cat']:<14} {p:<55} {r}")

## 14. Salvamento dos Resultados

In [ ]:
# Salva modelo treinado
model_avaliacao.save_pretrained('dompi_qwen25_final')
tokenizer.save_pretrained('dompi_qwen25_final')
print('Modelo salvo em dompi_qwen25_final')

# Salva métricas
resultados = {
    'modelo': MODEL_NAME,
    'dataset': 'gutoportelaa/DOMPI-2025',
    'territorios': TERRITORIOS,
    'n_treino': len(textos_treino),
    'n_avaliacao': len(textos_eval),
    'hiperparametros': {
        'epocas': args.num_train_epochs,
        'batch_efetivo': args.per_device_train_batch_size * args.gradient_accumulation_steps,
        'learning_rate': args.learning_rate,
        'max_length': MAX_LEN
    },
    'metricas_antes': metricas_antes,
    'metricas_depois': metricas_depois,
    'delta': {
        'entropia_cruzada': round(d_ec, 4),
        'perplexidade': round(d_ppl, 2),
        'acuracia_tokens_pp': round(d_acc, 2)
    }
}

with open('resultados_metricas.json', 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print('Métricas salvas em resultados_metricas.json')

with open('benchmark_dompi25.json', 'w', encoding='utf-8') as f:
    json.dump(benchmark, f, ensure_ascii=False, indent=2)
print('Benchmark salvo em benchmark_dompi25.json')

print('\n=== RESUMO FINAL ===')
print(f"Modelo base     -> PPL: {metricas_antes['perplexidade']} | EC: {metricas_antes['entropia_cruzada']} | Acc: {metricas_antes['acuracia_tokens']}%")
print(f"Modelo treinado -> PPL: {metricas_depois['perplexidade']} | EC: {metricas_depois['entropia_cruzada']} | Acc: {metricas_depois['acuracia_tokens']}%")

Modelo salvo em ./dompi_gpt2_final
Métricas salvas em resultados_metricas.json
Benchmark salvo em benchmark_dompi25.json

=== RESUMO FINAL ===
Modelo base     -> PPL: 50.71 | EC: 3.9262 | Acc: 35.81%
Modelo treinado -> PPL: 6.07 | EC: 1.8038 | Acc: 64.08%


In [ ]:
import shutil, os

os.makedirs('q1_saida', exist_ok=True)
for f in ['resultados_metricas.json', 'benchmark_dompi25.json', 'resultados_benchmark_base.json']:
    if os.path.exists(f):
        shutil.copy(f, os.path.join('q1_saida', f))

shutil.copytree('dompi_qwen25_final', 'q1_saida/dompi_qwen25_final', dirs_exist_ok=True)
shutil.make_archive('q1_resultados', 'zip', 'q1_saida')
print('Resultados em q1_resultados.zip (baixe pelo painel de saída do Kaggle/Colab).')